In [1]:
import json
from pprint import pprint
import torch
import os
import ast
import pandas as pd
from openai import OpenAI
import concurrent.futures



In [2]:
# read csv file

df = pd.read_csv('o4mini_annotation_final.csv')
df.head()


,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,t_result,t_corrected_version,t_category,t_notes,h_result,...,llm_errors,num_groups,agreement,text_length,llm_result_final,issues,llm_a1_agreement,llm_a2_agreement,a1_a2_agreement,llm_final_agreement
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[ \n { \n ""Chest pain"": [""non stop chest...","[\n {\n ""Chest pain"": [\n ""non stop c...",1,NaN,NaN,NaN,1,...,NaN,4.0,TRUE,110,"[\n {\n ""Chest pain"": [\n ""non stop c...",[],TRUE,TRUE,TRUE,TRUE
1,1527339,"cough, nasal congestion, runny nose, cold swea...","['Cold sweat', 'Cough', 'Fatigue', 'Hypogeusia...","""No temporal information found""",No temporal information found,1,NaN,NaN,NaN,1,...,NaN,NaN,TRUE,18,No temporal information found,[],TRUE,TRUE,TRUE,TRUE
2,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Headache"": [""none""],\n ""Immedi...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",Symptoms with no original mention should be gr...,NaN,0,...,NaN,2.0,FALSE,13,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...","['llm formatting issue', 'incorrect grouping',...",FALSE,FALSE,TRUE,FALSE
3,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [""injecti...","[\n {\n ""Vaccination site pain"": [\n ...",1,NaN,NaN,NaN,0,...,NaN,2.0,0,260,"[\n {\n ""Vaccination site pain"": [\n ...",[],TRUE,0,0,TRUE
4,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [""patient fainted""],\n ...","[\n {\n ""Syncope"": [\n ""patient faint...",1,NaN,NaN,NaN,1,...,NaN,2.0,1,118,"[\n {\n ""Syncope"": [\n ""patient faint...",[],1,1,1,1


In [3]:
# get columns from df as new df
data = df[['vaers_id','symptom_text', 'symptom_list','llm_result','llm_result_formatted','final_corrected_version','final_result']]
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 6000 non-null   int64 
 1   symptom_text             6000 non-null   object
 2   symptom_list             6000 non-null   object
 3   llm_result               6000 non-null   object
 4   llm_result_formatted     6000 non-null   object
 5   final_corrected_version  896 non-null    object
 6   final_result             6000 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 328.2+ KB


In [4]:
# Gold timeline = human-corrected version when one exists, otherwise the accepted
# o4-mini result. A report belongs to MedTempo-T unless that gold timeline is
# "No temporal information found".
data['gold_timeline'] = data['final_corrected_version'].where(
    data['final_corrected_version'].notna(), data['llm_result_formatted'])
data['no_temporal_info'] = data['gold_timeline'].astype(str).str.lower().str.contains('no temporal information')
print(data['no_temporal_info'].value_counts())
print(data.info())
# # get rows where no_temporal_info is False as new df
# data_with_temporal_info = data[data['no_temporal_info'] == False]
# data_with_temporal_info.info()
# # we have to remove final_result value 2 from data_with_temporal_info
# data_with_temporal_info = data_with_temporal_info[data_with_temporal_info['final_result'] != 2]
# data_with_temporal_info.info()

no_temporal_info
False    3794
True     2206
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 6000 non-null   int64 
 1   symptom_text             6000 non-null   object
 2   symptom_list             6000 non-null   object
 3   llm_result               6000 non-null   object
 4   llm_result_formatted     6000 non-null   object
 5   final_corrected_version  896 non-null    object
 6   final_result             6000 non-null   int64 
 7   no_temporal_info         6000 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 334.1+ KB
None


/tmp/ipykernel_625492/1703969331.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['no_temporal_info'] = data.apply(lambda row:


In [5]:
# # find data with no temporal info
# data_no_temporal_info = data[data['no_temporal_info'] == True]
# data_no_temporal_info = data_no_temporal_info[data_no_temporal_info['final_result'] != 2]

# data_no_temporal_info.info()

In [6]:
# # take 50 each from data_with_temporal_info and data_no_temporal_info
# data_sampled_with_temporal_info = data_with_temporal_info.sample(n=200, random_state=67)
# data_sampled_no_temporal_info = data_no_temporal_info.sample(n=100, random_state=67)
# # concatenate the two dataframes
# data_sampled = pd.concat([data_sampled_with_temporal_info, data_sampled_no_temporal_info])
# data_sampled.info()
# # save to csv
# # data_sampled.to_csv('sample_40.csv', index=False)

In [7]:
# # now lets take 40 random samples from data_with_temporal_info with random_state=2
# sampled_data = data_no_temporal_info 
# sampled_data.info()


In [8]:
eg = data[['vaers_id','symptom_text', 'symptom_list', 'no_temporal_info']].copy()
#eg.to_csv('openai_gold_with_temporal.csv', index=False)

In [9]:
eg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   vaers_id          6000 non-null   int64 
 1   symptom_text      6000 non-null   object
 2   symptom_list      6000 non-null   object
 3   no_temporal_info  6000 non-null   bool  
dtypes: bool(1), int64(1), object(2)
memory usage: 146.6+ KB


In [10]:
# convert no_temporal_info to reverse boolean has_temporal_info
eg['has_temporal_info'] = ~eg['no_temporal_info']
# eg = eg.sample(n = 100, random_state=67)
eg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   vaers_id           6000 non-null   int64 
 1   symptom_text       6000 non-null   object
 2   symptom_list       6000 non-null   object
 3   no_temporal_info   6000 non-null   bool  
 4   has_temporal_info  6000 non-null   bool  
dtypes: bool(2), int64(1), object(2)
memory usage: 152.5+ KB


In [11]:
import openai 
import time 
import random
import os
# chatgpt results
openai_api_key = os.environ["OPENAI_API_KEY"]

client = OpenAI(api_key=openai_api_key)


def llm(prompt, retries=5, delay=2, backoff=2):
    for attempt in range(retries):
        try:
            kwargs = dict(
            model="gpt-4.1",
            input=prompt,
            temperature= 0,
            # max_output_tokens=512
            )
            response = client.responses.create(**kwargs)
            return response.output_text
        except openai.RateLimitError:
            print(f"RateLimitError. Retrying in {delay} sec...")
        except openai.Timeout:
            print(f"Timeout. Retrying in {delay} sec...")
        except Exception as e:
            print(f"Other error: {str(e)}. Retrying in {delay} sec...")
        time.sleep(delay)
        delay *= backoff + random.uniform(0, 1)  # jitter to avoid spikes
    return "Error: Failed after retries"
    
def clean_and_parse_json(text):
    try:
        if isinstance(text, str):
            cleaned = text.replace('\n', '')
            return json.loads(cleaned)
    except json.JSONDecodeError:
        return None

    return None

def validate_output_symptoms(row):
    if isinstance(row['llm_result_cleaned'], list):
        output_symptoms = [key for dic in row['llm_result_cleaned'] for key in dic.keys()]
        input_symptoms = ast.literal_eval(row['symptom_list'])
        return set(output_symptoms) == set(input_symptoms)
    return False

def get_ordered_symptoms_flatten(symptom_list):
    symptoms = []
    if isinstance(symptom_list, list):
        for sym_dict in symptom_list:
            symptoms.extend(list(sym_dict.keys()))
    return symptoms

def get_ordered_symptoms(symptom_list):
    symptoms = []
    if isinstance(symptom_list, list):
        for sym_dict in symptom_list:
            symptoms.append(list(sym_dict.keys()))
    return symptoms

# def call_llm(row):
    
#     prompt = temporal_id_prompt_tpl.format(  
#         suggest=row['symptom_list'],
#         example=example,
#         text=row['symptom_text']
        
#     )
#     return llm(prompt)

In [12]:
# =========================
# Temporal pattern gating (binary classifier)
# =========================
from string import Template

temporal_id_prompt_tpl = Template(r"""
You are an expert medical analyst. Your task is to determine if there is a temporal progression of different adverse events in the clinical notes. 

Definition: Temporal progression means a sequence where a symptom not previously mentioned appears after a previously mentioned symptom. The new symptoms must be distinct from any symptoms already listed in the clinical notes. 

What does not constitute temporal progression: 
- Concurrent Onset: All symptoms begin at the same time. 
- Symptom Resolution: A symptom disappears or improves. 
- Fluctuating Severity: A symptom gets better and worse. 
- Persistence: A symptom persists. 
- Inferred Timelines: Timelines based on symptom durations alone. For example, "Headache for 4 days, fever for 3 days" does not demonstrate temporal progression. 
- Assumptions: Any assumptions about the order of events if not directly stated in the text. 
- Re-emergence of Previous Symptoms: Re-emergence or continuation of previously reported symptoms, even with changes in severity, does not constitute temporal progression. The sequence must involve entirely new symptoms. 

Focus: Focus on whether different adverse events appear at different times, as explicitly stated in the clinical notes. Do not consider changes to the same adverse event over time. Pay close attention to whether the clinical notes introduce a symptom that has not been mentioned previously. 

Example of what does not constitute temporal progression: "Patient reported fatigue for 5 days and a cough for 2 days, then the fatigue returned." The return of fatigue is not a new event. Another example: "Patient reported headache and nausea, then the headache persisted the next day. This is not temporal progression." 

Clinical Notes:
$note

Output:
- If a temporal progression of different adverse events is identified, return exactly: Temporal information found
- If no temporal progression of different adverse events is found, return exactly: No temporal information found
""")

def parse_temporal_gate_output(raw: str):
    if not isinstance(raw, str):
        return None, None
    t = raw.strip()
    if "Temporal information found" in t:
        return "Temporal information found", True
    if "No temporal information found" in t:
        return "No temporal information found", False
    return None, None

def call_temporal_gate(row):
    note = row["symptom_text"]
    prompt = temporal_id_prompt_tpl.substitute(note=note if note is not None else "")
    raw = llm(prompt)
    label, flag = parse_temporal_gate_output(raw)
    return raw, label, flag


In [13]:
# =========================
# Run temporal gating in parts (e.g., batch=50 like your timeline runs)
# =========================

gate_df = eg  # change to whichever DataFrame you want to gate

GATE_START = 0
GATE_END = None               # e.g., 100 for a small test
GATE_BATCH_SIZE = 1000          # match your batch split
GATE_MAX_WORKERS = 4          # increase carefully if rate-limited

# Output columns
for col in ["temporal_gate_raw", "temporal_gate_label", "has_temporal_info_model"]:
    if col not in gate_df.columns:
        gate_df[col] = None

end = len(gate_df) if GATE_END is None else min(GATE_END, len(gate_df))

for i in range(GATE_START, end, GATE_BATCH_SIZE):
    j = min(i + GATE_BATCH_SIZE, end)
    print(f"[gate] Processing rows {i}..{j-1}")

    batch = gate_df.iloc[i:j]
    with concurrent.futures.ThreadPoolExecutor(max_workers=GATE_MAX_WORKERS) as executor:
        outputs = list(executor.map(call_temporal_gate, [row for _, row in batch.iterrows()]))

    gate_df.loc[batch.index, "temporal_gate_raw"] = [o[0] for o in outputs]
    gate_df.loc[batch.index, "temporal_gate_label"] = [o[1] for o in outputs]
    gate_df.loc[batch.index, "has_temporal_info_model"] = [o[2] for o in outputs]

    # checkpoint
    gate_df.to_csv(f"temporal_gate_final_part_{i}.csv", index=False)
gate_df.to_csv(f"temporal_gate_final.csv", index=False)
print("Done. Missing has_temporal_info_model:", gate_df["has_temporal_info_model"].isna().sum())


[gate] Processing rows 0..999
[gate] Processing rows 1000..1999
[gate] Processing rows 2000..2999
[gate] Processing rows 3000..3999
[gate] Processing rows 4000..4999
[gate] Processing rows 5000..5999
Done. Missing has_temporal_info_model: 0


In [14]:
gate_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 6000 non-null   int64 
 1   symptom_text             6000 non-null   object
 2   symptom_list             6000 non-null   object
 3   no_temporal_info         6000 non-null   bool  
 4   has_temporal_info        6000 non-null   bool  
 5   temporal_gate_raw        6000 non-null   object
 6   temporal_gate_label      6000 non-null   object
 7   has_temporal_info_model  6000 non-null   object
dtypes: bool(2), int64(1), object(5)
memory usage: 293.1+ KB


In [15]:
# need to convert has_temporal_info_model to boolean by using literal_eval
gate_df['has_temporal_info_model'] = gate_df['has_temporal_info_model'].apply(lambda x: ast.literal_eval(str(x)) if pd.notna(x) else x)
gate_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 6000 non-null   int64 
 1   symptom_text             6000 non-null   object
 2   symptom_list             6000 non-null   object
 3   no_temporal_info         6000 non-null   bool  
 4   has_temporal_info        6000 non-null   bool  
 5   temporal_gate_raw        6000 non-null   object
 6   temporal_gate_label      6000 non-null   object
 7   has_temporal_info_model  6000 non-null   bool  
dtypes: bool(3), int64(1), object(4)
memory usage: 252.1+ KB


In [16]:
gate_df.head()

,vaers_id,symptom_text,symptom_list,no_temporal_info,has_temporal_info,temporal_gate_raw,temporal_gate_label,has_temporal_info_model
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...",False,True,Temporal information found,Temporal information found,True
1,1527339,"cough, nasal congestion, runny nose, cold swea...","['Cold sweat', 'Cough', 'Fatigue', 'Hypogeusia...",True,False,No temporal information found,No temporal information found,False
2,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...",False,True,Temporal information found,Temporal information found,True
3,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...",False,True,Temporal information found,Temporal information found,True
4,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...",False,True,Temporal information found,Temporal information found,True


In [17]:
# calculate precision, recall, f1 for has_temporal_info_model against has_temporal_info
from sklearn.metrics import precision_score, recall_score, f1_score
precision = precision_score(gate_df['has_temporal_info'], gate_df['has_temporal_info_model'])
recall = recall_score(gate_df['has_temporal_info'], gate_df['has_temporal_info_model'])
f1 = f1_score(gate_df['has_temporal_info'], gate_df['has_temporal_info_model'])
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

Precision: 0.9811, Recall: 0.5894, F1: 0.7364
